In [0]:
from databricks.sdk import WorkspaceClient
from datetime import datetime, timezone

DEFAULT_AS_OF = spark.sql(
    "SELECT date_format(from_utc_timestamp(current_timestamp(), 'America/Toronto'), 'yyyy-MM-dd') AS d"
).first()["d"]

dbutils.widgets.text("catalog", "serverless_nv91sa_catalog")
dbutils.widgets.text("team", "team1")
dbutils.widgets.text("pipeline_id", "")
dbutils.widgets.text("as_of_date", DEFAULT_AS_OF)

catalog = dbutils.widgets.get("catalog")
team = dbutils.widgets.get("team")
pipeline_id = dbutils.widgets.get("pipeline_id")

freshness = spark.sql(
    f"SELECT unix_timestamp(current_timestamp()) - unix_timestamp(MAX(order_ts)) AS age_seconds FROM {catalog}.{team}.gold_order_margin"
).first()["age_seconds"]

pipeline_ok = False
if pipeline_id:
    workspace = WorkspaceClient()
    # TODO: confirm the exact updates API surface if this check is promoted beyond workshop use.
    updates = list(workspace.pipelines.list_updates(pipeline_id=pipeline_id, max_results=1))
    pipeline_ok = bool(updates) and str(updates[0].state).upper().endswith("COMPLETED")

status = "PASS" if pipeline_ok and freshness is not None and freshness <= 900 else "FAIL"
metric_value = float(freshness or -1)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.ops.milestones (team STRING, round INT, status STRING, metric_name STRING, metric_value DOUBLE, ts TIMESTAMP)")
spark.sql(
    f"INSERT INTO {catalog}.ops.milestones VALUES ('{team}', 1, '{status}', 'gold_order_margin_age_seconds', {metric_value}, current_timestamp())"
)

print({"status": status, "pipeline_ok": pipeline_ok, "freshness_seconds": freshness})
